# RQ2 · Health spending and life expectancy  -  with gender
### Does higher public health expenditure predict higher life expectancy, and are the returns different for women vs men and across income groups?

**Design.**
- Unit of analysis: country × year.
- Sample: all countries (drop aggregates), 2000-2022 (health expenditure
  series starts consistently in 2000).
- Outcomes: `SP.DYN.LE00.FE.IN` (female life expectancy), `SP.DYN.LE00.MA.IN` (male), and their **gap** (`female − male`).
- Regressor of interest: `log(current health expenditure per capita, US$)`.
- Controls: `log(GDP per capita)`, `log(population)`, physicians per 1,000.
- Fixed effects: country + year.
- SEs: clustered at country.

**Why this design?** Health spending and health outcomes have obvious
reverse causality (sicker populations demand more spending). Two-way FE
handles level differences and common time shocks. To assess whether
gains are shared, we run the regression **separately** for female and
male life expectancy and formally test the difference of coefficients.
The gender-gap regression uses the gap itself as the outcome so we can
speak directly about closing/widening.

**Compute footprint.** ~200 countries × 23 years ≈ 4,600 rows. Local run
in seconds. OSG version runs the 5 income-group × gender permutations
with bootstrap SEs.


In [ ]:
import sys, warnings
sys.path.insert(0, "..")
warnings.simplefilter("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.wb_client import fetch_indicators, fetch_country_metadata, RQ2_INDICATORS
from src.regression import fixed_effects_panel


## 1. Pull the panel

In [ ]:
panel = fetch_indicators(
    indicators=list(RQ2_INDICATORS.keys()),
    countries="all",
    start=2000, end=2022,
)
meta = fetch_country_metadata()
real = set(meta.loc[~meta["is_aggregate"], "iso3"])
panel = panel[panel["iso3"].isin(real)].copy()
panel = panel.merge(meta[["iso3", "region", "income"]], on="iso3", how="left")

print(f"{len(panel):,} country-years × {panel['iso3'].nunique():,} countries")


## 2. Build regressors and outcomes

In [ ]:
panel["log_health_pc"] = np.log(panel["SH.XPD.CHEX.PC.CD"].replace(0, np.nan))
panel["log_gdp_pc"]    = np.log(panel["NY.GDP.PCAP.KD"].replace(0, np.nan))
panel["log_pop"]       = np.log(panel["SP.POP.TOTL"].replace(0, np.nan))
panel = panel.rename(columns={
    "SP.DYN.LE00.FE.IN": "le_female",
    "SP.DYN.LE00.MA.IN": "le_male",
    "SP.DYN.LE00.IN":    "le_total",
    "SH.MED.PHYS.ZS":    "physicians_per_1k",
})
panel["le_gap"] = panel["le_female"] - panel["le_male"]
panel[["iso3", "year", "log_health_pc", "le_female", "le_male", "le_gap"]].head()


## 3. Descriptive: the female-male gap over time

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
by_income = panel.groupby(["year", "income"])["le_gap"].mean().unstack()
by_income.plot(ax=ax, marker="o")
ax.set_ylabel("Female − Male life expectancy (years)")
ax.set_title("Gender gap in life expectancy by income group")
ax.grid(alpha=0.3)
ax.legend(title="", fontsize=8)
fig.tight_layout()


## 4. Main regression  -  total life expectancy

In [ ]:
main = fixed_effects_panel(
    panel,
    dependent="le_total",
    regressors=["log_health_pc", "log_gdp_pc", "physicians_per_1k", "log_pop"],
    entity_effects=True, time_effects=True,
    name="RQ2 main  -  Health spending → total life expectancy",
)
print(main.as_markdown())


## 5. By gender  -  is the return to health spending the same for women and men?

In [ ]:
for outcome in ["le_female", "le_male", "le_gap"]:
    r = fixed_effects_panel(
        panel,
        dependent=outcome,
        regressors=["log_health_pc", "log_gdp_pc", "physicians_per_1k", "log_pop"],
        entity_effects=True, time_effects=True,
        name=f"RQ2  -  Health → {outcome}",
    )
    row = r.coef_table.loc["log_health_pc"]
    print(f"{outcome:12s}  β(log_health_pc) = {row['coef']:+.3f}  (SE {row['stderr']:.3f}, p={row['pvalue']:.3g})")


## 6. Interacted specification  -  is the effect stronger at low incomes?

Interaction of `log_health_pc` with an indicator for below-median GDP per
capita in the sample. A negative coefficient on the interaction means
richer countries get less bang per health dollar.


In [ ]:
median_gdp = panel["log_gdp_pc"].median()
panel["low_income_ind"] = (panel["log_gdp_pc"] < median_gdp).astype(float)
panel["health_x_low"]   = panel["log_health_pc"] * panel["low_income_ind"]

res_i = fixed_effects_panel(
    panel,
    dependent="le_total",
    regressors=["log_health_pc", "health_x_low", "log_gdp_pc", "physicians_per_1k", "log_pop"],
    entity_effects=True, time_effects=True,
    name="RQ2 interacted  -  health × low-income",
)
print(res_i.as_markdown())


## 7. Save

In [ ]:
from pathlib import Path
Path("../results").mkdir(exist_ok=True)
main.coef_table.to_csv("../results/rq2_health_main_coefs.csv")
res_i.coef_table.to_csv("../results/rq2_health_interacted_coefs.csv")
